In [ ]:
import numpy as np
import healpy as hp
import pandas as pd
import geopandas as gpd

from pyproj import CRS, Transformer
from healpix_geo.nested import healpix_to_lonlat
from healpix_geo.nested import lonlat_to_healpix

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

In [ ]:
def get_points(NSIDE):
    NPIX = hp.nside2npix(NSIDE)
    points = np.array(
        hp.pix2vec(NSIDE, range(NPIX), nest=True)
    ).T

    df = pd.DataFrame(points, columns=('x', 'y', 'z'))
    df['neighbour'] = 'none'

    return df

def get_patches(NSIDE):
    NPIX = hp.nside2npix(NSIDE)

    pixels = np.arange(NPIX)
    patches = np.array(
        [hp.boundaries(NSIDE, i, nest=True).T for i in pixels]
    ).reshape(-1, 3)

    df = pd.DataFrame(patches, columns=('x', 'y', 'z'))
    df['neighbour'] = 'none'

    return df

In [ ]:
kaz = gpd.read_file('data/geoBoundaries-KAZ-ADM0_simplified.geojson')
ukr = gpd.read_file('data/geoBoundaries-UKR-ADM0_simplified.geojson')
rus = gpd.read_file('data/geoBoundaries-RUS-ADM0_simplified.geojson')
blr = gpd.read_file('data/geoBoundaries-BLR-ADM0_simplified.geojson')
borders = pd.concat((kaz, ukr, rus, blr)).reset_index(drop=True)
borders = borders.to_crs(epsg=4326)
borders

In [ ]:
fig = px.choropleth(
    borders,
    geojson=borders.geometry,
    locations=borders.shapeISO,
    locationmode='ISO-3',
    color='shapeName',
)
fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
)
fig.show()

In [ ]:
NSIDE = 32
healpix = get_points(NSIDE)

In [ ]:
NPIX = hp.nside2npix(NSIDE)
ipix = np.arange(NPIX)
depth = np.sqrt(NSIDE).astype(int)

In [ ]:
lon, lat = healpix_to_lonlat(ipix, depth=depth, ellipsoid='WGS84')
lon = (lon + 180) % 360 - 180

In [ ]:
gdf = gpd.GeoDataFrame(
    healpix.drop(['neighbour'], axis=1),
    geometry=gpd.points_from_xy(
        lon, lat
    ), crs='EPSG:4326'
)
gdf

In [ ]:
airports = pd.read_csv('data/airports.csv')
airports = gpd.GeoDataFrame(
    airports.drop(['GeoPointLat', 'GeoPointLong'], axis=1),
    geometry=gpd.points_from_xy(
        *airports[['GeoPointLong', 'GeoPointLat']].values.T
    ), crs='EPSG:4326'
)
airports[airports.ICAO.isin(['KJFK', 'UAAA'])]

In [ ]:
gdf['avoid'] = np.array([
    gdf.geometry.within(shape)
    for _, shape
    in borders[borders.shapeISO!='KAZ'].geometry.items()
]).sum(axis=0)

avoid_neighbours = np.unique(
    hp.get_all_neighbours(
        NSIDE, gdf[gdf.avoid==1].index.values, nest=True
    )
)
gdf.loc[avoid_neighbours, 'avoid'] = 1

In [ ]:
fig = go.Figure()

trace = go.Choropleth(
    geojson=borders.geometry.__geo_interface__,
    locations=borders.shapeISO,
    z=list(range(len(borders))),
    locationmode='ISO-3',
    featureidkey='properties.shapeISO',
    colorscale=px.colors.qualitative.Plotly[:len(borders)],
    text=borders['shapeName'],
    showscale=False,
)
fig.add_trace(trace)

for flag, color in zip(
    [0, 1],
    ['black', 'white']
):
    points = go.Scattergeo(
        lon=gdf[gdf.avoid==flag].geometry.x,
        lat=gdf[gdf.avoid==flag].geometry.y,
        mode='markers',
        marker=dict(size=2, color=color),
    )
    fig.add_trace(points)

route = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(route)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
)
fig.show()

In [ ]:
gdf

In [ ]:
arr = np.array([3, 5, 2])
arr[[1, 1]] = 0
arr

In [ ]:
arr.choose()

In [ ]:
dim = len(gdf.index)
connection_mtx = np.diag(np.ones(dim))

neighbours = hp.get_all_neighbours(
    NSIDE, range(dim), nest=True
).T

for idx, neighbour_ids in enumerate(neighbours):
    neighbour_ids = np.where(
        neighbour_ids > 0,
        neighbour_ids,
        idx,
    )
    connection_mtx[idx, neighbour_ids] = 1

for idx in gdf[gdf.avoid==1].index:
    connection_mtx[idx] *= 0
    connection_mtx[idx, idx] = 1

In [ ]:
obstacle_mtx = 1 - np.diag(gdf.avoid)

In [ ]:
start = lonlat_to_healpix(
    airports[airports.ICAO=='UAAA'].geometry.x,
    airports[airports.ICAO=='UAAA'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]
end = lonlat_to_healpix(
    airports[airports.ICAO=='KJFK'].geometry.x,
    airports[airports.ICAO=='KJFK'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]

In [ ]:
import networkx as nx

In [ ]:
G = nx.from_numpy_array(connection_mtx)

In [ ]:
route = nx.astar_path(
    G, start, end,
)

In [ ]:
fig = go.Figure()

trace = go.Choropleth(
    geojson=borders.geometry.__geo_interface__,
    locations=borders.shapeISO,
    z=list(range(len(borders))),
    locationmode='ISO-3',
    featureidkey='properties.shapeISO',
    colorscale=px.colors.qualitative.Plotly[:len(borders)],
    text=borders['shapeName'],
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf.iloc[route].geometry.x,
    lat=gdf.iloc[route].geometry.y,
    mode='markers',
    marker=dict(size=3, color='black'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
)
fig.show()